# Build and Test a Local Image

This notebook builds a small, learner-owned web-server image from the files under `packages/docker_exercises/`. Run every Docker command from an authorized base-station terminal. The container name, image tag, volume name, and host port used here are reserved for this LX.

## Inspect the build context

A __build context__ is the directory Docker receives when building an image. From the LX repository root, save that location and inspect the supplied exercise files:

```bash
LX_ROOT="$PWD"
ls packages/docker_exercises
cat packages/docker_exercises/Dockerfile
cat packages/docker_exercises/server.py
cat packages/docker_exercises/.dockerignore
```

The `Dockerfile` describes how to build an image. `server.py` is a small web server, and `.dockerignore` excludes unnecessary files from the build context. Read the files before building so you understand what Docker will copy into the image.

The supplied Dockerfile uses these common instructions:

| Instruction | Purpose |
| --- | --- |
| `FROM` | Selects the base image. |
| `WORKDIR` | Sets the working directory for later instructions and the running program. |
| `COPY` | Copies a file from the build context into the image. |
| `ENV` | Sets an environment variable used by the running program. |
| `USER` | Selects the default account for the running container. |
| `EXPOSE` | Documents the port used by the application. It does not publish the port. |
| `ENTRYPOINT` | Sets the executable used when the container starts. |
| `CMD` | Supplies default arguments to the entrypoint. |

## Build a local image

Build the image with a name and version that identify this practice work:

```bash
docker build --tag lx-docker-hello:0.1 packages/docker_exercises
docker image ls lx-docker-hello
docker image inspect --format '{{.Config.User}}' lx-docker-hello:0.1
```

`docker build` reads `packages/docker_exercises/Dockerfile` and creates a local image. `--tag` assigns the image name `lx-docker-hello` and the version-like tag `0.1`. The inspection command should report the non-root account configured by the Dockerfile.

The first build can download the explicitly tagged base image. A later build can reuse cached, unchanged instructions. If you edit `server.py` and rebuild, Docker can reuse the earlier `FROM` and `WORKDIR` instructions, then applies the changed `COPY` instruction and the later image configuration.

## Publish a local web port

Start the web server and publish it only to the base station's loopback address:

```bash
docker run --detach --name lx-docker-web \
  --publish 127.0.0.1:8088:8080 \
  lx-docker-hello:0.1
docker container ls --filter 'name=lx-docker-web'
docker container port lx-docker-web
```

The application listens on port `8080` inside the container. `--publish 127.0.0.1:8088:8080` forwards base-station port `8088` to that container port, but only for connections made from the base station itself. The first port is the host port; the second is the container port.

`localhost` is interpreted in the network context of the process making a request. The `curl` command below runs on the base station, so `127.0.0.1:8088` names the base-station side of the published mapping. Inside the web container, `127.0.0.1` refers to that container itself, not the base station.

`EXPOSE 8080` in a Dockerfile does not publish the port by itself. Conversely, omitting `127.0.0.1:` would normally publish the port on all host network interfaces. Bind to loopback in this LX so the practice web server is not offered to the local network.

The supplied server binds to `0.0.0.0` inside its own container. That means it accepts traffic sent to the container's network address; it does not make the service public on the base station. Docker's explicit `--publish` rule controls the host-side access.

## Request the application response

Use a targeted HTTP request to verify that the application, not just its TCP port, responds:

```bash
curl --fail --silent --retry 5 --retry-all-errors --max-time 10 http://127.0.0.1:8088/
docker logs lx-docker-web
```

`curl` requests the page through the published loopback port. `--fail` returns a nonzero status for HTTP error responses, `--silent` keeps temporary connection failures out of the successful output, and the bounded retry options allow the newly started server a few seconds to become ready. `--max-time 10` prevents any one request from waiting too long. `docker logs` then shows the server's startup and request messages.

If the request fails, inspect only `lx-docker-web` with `docker container ls --all --filter 'name=lx-docker-web'` and `docker logs lx-docker-web`. Do not publish another port, change the Docker context, or disable network protections to work around the problem.

## Further reading

Docker's official guides explain [writing a Dockerfile](https://docs.docker.com/get-started/docker-concepts/building-images/writing-a-dockerfile/) and [publishing ports](https://docs.docker.com/get-started/docker-concepts/running-containers/publishing-ports/).

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
